# Early Intervention & Developmental Screening Triage—Load & Explore

**Agents for Impact 2026 · Early Intervention Triage (`A4I2026-challenge-early-intervention`)**

You are building an agent for a **Regional IDEA Part C Intake & Quality Coordinator**—internal state and regional early-childhood staff who route newly referred infants and toddlers (aged `0–35 months`) from primary-care developmental screening into multidisciplinary evaluation and active therapy **before the federal 45-day Individualized Family Service Plan (IFSP) clock expires** (`34 CFR § 303.310`). Your tool supports internal care-coordination and network quality teams; it does **not** diagnose children or replace clinical judgment.

This notebook sets up your BigQuery **Enterprise graph reservation** (`Section 1b`), loads and validates your five-table BigQuery foundation, and stops right before declaring your **BigQuery Property Graph** (`CREATE PROPERTY GRAPH`) and **ISO GQL** multi-hop traversals (`Section 13`)—that part is yours.

## Plain-words data honesty — stage-safe `[Synthetic]` provider network

Assigning simulated waitlists, fax drop-offs, or federal 45-day IDEA Part C compliance breaches to real named pediatricians or real clinic street addresses would defame real providers on stage. Therefore, **every organization and referral edge in the care network is 100% `[Synthetic]`**, anchored to real US Census tract demographics and statewide HRSA survey baselines:

| Table | Grain | Source & Provenance | What it tells you |
|---|---|---|---|
| `acs_under5_tract_demographics` | 1 row / census tract | **Real public data** (US Census ACS 5-Year) | Children under 5 (`children_under_5_count`), poverty, zero-vehicle households, `tract_geom`, and post-output `audit_*` shares |
| `nsch_developmental_screening_survey` | 1 row / state × demographic stratum | **`[Modeled]` statewide strata** calibrated to **HRSA National Survey of Children's Health (NSCH)** | Standardized screening rates (`9–35m`) and follow-up rates across FPL income bands and household language (**`geographic_level = 'Statewide'` only!**) |
| `idea_part_c_agency_performance` | 1 row / regional Part C lead agency | **`100% [Synthetic]`** (`SYN-PARTC-*`) calibrated to US Dept. of Education OSEP Part C baselines | Regional referral intake volume, completed evaluations, 45-day timely IFSP counts, capacity vs. family delays, and `intake_geom` |
| `pediatric_and_ei_provider_directory` | 1 row / `CareNode` organization | **`100% [Synthetic]`** (`SYN-PROV-*`, `30`/state) | Homogeneous graph vertices across 4 `node_role`s (`Screening_Pediatric_Clinic`, `Part_C_Intake_Coordinator`, `Multidisciplinary_Eval_Hub`, `EI_Therapy_Vendor`), Medicaid/bilingual flags, waitlists, and `provider_geom` |
| `deidentified_part_c_referral_logs` | 1 row / `ReferralHandoff` edge | **`100% [Synthetic]`** (`149` handoffs across `44` `case_chain_id`s/state) | Directed `2..4` hop referral and re-referral bounce episodes (`hop_sequence`), `paper_fax` vs. `ehr_electronic_portal` vs. `phone_voicemail`, `handoff_wait_days`, and `dropoff_rate_pct` |

## The four lanes, and how to spend your 4.5 hours

- **Data lane.** Run this notebook (~2–3 minutes including `Section 1b` reservation propagation on a brand-new project), then build `a4i_early_intervention.ReferralGraph` and your variable-length `-{2,4}->` ISO GQL queries from Section 13.
- **Agent lane.** Build an intake & network-audit agent that (a) diagnoses multi-hop re-referral bounce chains pushing children past 45 days and (b) recommends fast-track Medicaid/bilingual routing alternatives.
- **Front-end lane.** Build an interactive referral-chain & neighborhood gap dashboard against the five BigQuery tables immediately; wire it to your live Cloud Run / Agent Runtime endpoint once deployed.
- **Story lane.** Graded asynchronously from your **GitHub repo**, **Google Skills project**, **pitch deck**, and **3-minute video**. State plainly that `SYN-PROV-*` / `SYN-PARTC-*` are `100% [Synthetic]`, explain why NSCH is statewide-only, and report your `audit_*` Equity Audit.

## Before you start

Two prompts appear the first time you open **Colab Enterprise** in a fresh Google Cloud project, and both are expected:

1. Opening Colab Enterprise prompts you to **enable required APIs**—accept them.
2. The Colab Enterprise header may display a second **"Enable APIs"** banner—enable that too.

Everything this notebook imports (`google-cloud-bigquery`, `pandas`, `pyarrow`, `requests`) is already pre-installed on the standard Colab Enterprise runtime. Do not `pip install` extra packages here.

**How to run:** Pick your `STATE` in the next cell (default: `"MS"` — Mississippi) and select **Runtime → Run all**.

## 1. Setup

In [ ]:
# ---- the one thing you change ------------------------------------------------
STATE    = "MS"                       # Choose from STATES: MS, AL, GA, TX, CA, NY
DATASET  = "a4i_early_intervention"   # BigQuery dataset created in your own GCP project
LOCATION = "US"                       # Keep in US (matches region-us reservation in 1b)
# ------------------------------------------------------------------------------

import contextlib
import io
import json
import os
from pathlib import Path
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from google.cloud import bigquery

STATES = {
    "MS": ("Mississippi", "28"),
    "AL": ("Alabama",     "01"),
    "GA": ("Georgia",     "13"),
    "TX": ("Texas",       "48"),
    "CA": ("California",  "06"),
    "NY": ("New York",    "36"),
}
if STATE not in STATES:
    raise ValueError(f"Unknown STATE={STATE!r}. Choose one of: {list(STATES)}")

STATE_NAME, STATE_FIPS = STATES[STATE]

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT:
    PROJECT = bigquery.Client().project
if not PROJECT:
    raise RuntimeError("Could not determine GCP project. Set GOOGLE_CLOUD_PROJECT.")

bq = bigquery.Client(project=PROJECT, location=LOCATION)
ds_ref = bigquery.Dataset(f"{PROJECT}.{DATASET}")
ds_ref.location = LOCATION
bq.create_dataset(ds_ref, exists_ok=True)

print(f"project     : {PROJECT}")
print(f"state       : {STATE_NAME} ({STATE}, state FIPS {STATE_FIPS})")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"pandas      : {pd.__version__}")

### Timing, two-tier validation, and resilient GeoParquet loading

In [ ]:
STEPS, CHECKS = {}, []
NOTEBOOK_START = time.time()

GITHUB_ORG = os.environ.get("A4I_GITHUB_ORG", "haggman")
REPO_NAME = "A4I2026-challenge-early-intervention"
SNAPSHOT_BUCKET = os.environ.get("A4I_SNAPSHOT_BUCKET", "class-demo")
SNAPSHOT_PREFIX = "a4i-2026/challenge-early-intervention"
UA = {"User-Agent": "A4I2026-challenge-ei-notebook"}


@contextlib.contextmanager
def step(name):
    t0 = time.time()
    print(f"[{name}] ...")
    try:
        yield
    finally:
        dt = time.time() - t0
        STEPS[name] = round(dt, 1)
        print(f"[{name}] {dt:.1f}s")


def _record(name, verdict, detail):
    for c in CHECKS:
        if c["check"] == name:
            c["verdict"], c["detail"] = verdict, detail
            return
    CHECKS.append({"check": name, "verdict": verdict, "detail": detail})


def check(name, ok, detail=""):
    _record(name, "PASS" if ok else "FAIL", detail)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f": {detail}" if detail else ""))
    return bool(ok)


def note(name, bad, total, detail="", limit=0.01, rows=None):
    frac = (bad / total) if total else 0.0
    verdict = "PASS" if bad == 0 else ("FAIL" if (total and frac > limit) else "WARN")
    msg = detail or (f"{bad:,} of {total:,}" if total else f"{bad:,}")
    _record(name, verdict, msg)
    print(f"  {verdict}  {name}: {msg}")
    if verdict != "PASS" and rows is not None:
        display(rows.head(10))
    return verdict


def safe_str(s, width=None):
    out = s.astype("string")
    if width:
        out = out.str.strip().str.zfill(width)
    return out.where(out.notna(), None)


def spread(series, label=""):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if s.empty:
        return {}
    p10, p50, p90 = s.quantile([0.10, 0.50, 0.90])
    out = {
        "n": int(s.size),
        "min": round(float(s.min()), 3),
        "p10": round(float(p10), 3),
        "median": round(float(p50), 3),
        "p90": round(float(p90), 3),
        "max": round(float(s.max()), 3),
    }
    if label:
        print(f"  {label:<32} " + "  ".join(f"{k}={v}" for k, v in out.items()))
    return out

In [ ]:
def _fetch_parquet_bytes(url, tries=4, timeout=60):
    delay = 1.5
    for attempt in range(tries):
        try:
            resp = requests.get(url, timeout=timeout, headers=UA)
        except requests.RequestException as exc:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — {type(exc).__name__}: {exc}") from exc
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code == 429 or 500 <= resp.status_code < 600:
            if attempt == tries - 1:
                raise RuntimeError(f"{url} — HTTP {resp.status_code} after {tries} tries")
            time.sleep(delay)
            delay *= 2
            continue
        if resp.status_code != 200:
            raise RuntimeError(f"{url} — HTTP {resp.status_code}")
        if resp.content[:4] != b"PAR1":
            raise RuntimeError(f"{url} — response is not a valid Parquet file")
        return resp.content
    raise RuntimeError(f"{url} — exhausted retries")


def read_starter_parquet(filename):
    if not filename or "/" in filename or ".." in filename:
        raise ValueError(f"Invalid starter filename: {filename!r}")
    data_dir = os.environ.get("A4I_DATA_DIR", "").strip()
    local_candidates = (
        ([Path(data_dir) / filename] if data_dir else [])
        + [Path("../data") / filename, Path("data") / filename, Path(filename)]
    )
    for path in local_candidates:
        if path.is_file():
            return pd.read_parquet(path)

    base_override = os.environ.get("A4I_DATA_BASE_URL", "").strip().rstrip("/")
    urls = ([f"{base_override}/{filename}"] if base_override else []) + [
        f"https://raw.githubusercontent.com/{GITHUB_ORG}/{REPO_NAME}/main/data/{filename}",
        f"https://storage.googleapis.com/{SNAPSHOT_BUCKET}/{SNAPSHOT_PREFIX}/starter/{filename}",
    ]
    errors = []
    for url in urls:
        try:
            return pd.read_parquet(io.BytesIO(_fetch_parquet_bytes(url)))
        except RuntimeError as exc:
            errors.append(str(exc))
    raise RuntimeError(f"Could not locate {filename} locally or remotely: {'; '.join(errors)}")


def _to_geoparquet_buffer(df, geom_col=None):
    table = pa.Table.from_pandas(df, preserve_index=False)
    if geom_col and geom_col in df.columns:
        geo_meta = {
            "version": "1.0.0",
            "primary_column": geom_col,
            "columns": {
                geom_col: {
                    "encoding": "WKB",
                    "geometry_types": [],
                    "crs": None,
                    "orientation": "counterclockwise",
                    "edges": "spherical",
                }
            },
        }
        meta = dict(table.schema.metadata or {})
        meta[b"geo"] = json.dumps(geo_meta).encode("utf-8")
        table = table.replace_schema_metadata(meta)
    buf = io.BytesIO()
    pq.write_table(table, buf)
    buf.seek(0)
    return buf


def load_table(df, table_name, geom_col=None, description=""):
    if df is None or not table_name:
        raise ValueError("df and table_name are required")
    ref = f"{PROJECT}.{DATASET}.{table_name}"
    buf = _to_geoparquet_buffer(df, geom_col=geom_col)
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    bq.load_table_from_file(buf, ref, job_config=cfg).result()
    tbl = bq.get_table(ref)
    if description:
        tbl.description = description
        bq.update_table(tbl, ["description"])
    print(f"  loaded {table_name:<38} {tbl.num_rows:>6,} rows")
    return tbl

### 1b. Provision a zero-baseline Enterprise reservation for BigQuery Graph

**Do not skip this cell.** BigQuery Property Graph queries (`GRAPH ... MATCH ...` and `GRAPH_TABLE`) require a reservation on the **Enterprise** or **Enterprise Plus** edition. Fresh Google Cloud projects default to on-demand pricing, which fails on the first `GRAPH` query with:

> *BigQuery Graph queries require a reservation with Enterprise or Enterprise Plus edition.*

The cell below creates an idempotent reservation (`a4i-graph`) in `region-us` with **`slot_capacity = 0` and `autoscale_max_slots = 50`**—meaning it costs **\$0 while idle** and borrows a handful of slots for a few seconds only when a graph query runs. Re-running is a fast no-op once in place.

In [ ]:
RESERVATION = "a4i-graph"
_region = f"region-{LOCATION.lower()}"
_res_path = f"`{PROJECT}.{_region}.{RESERVATION}`"
_asg_path = f"`{PROJECT}.{_region}.{RESERVATION}.{RESERVATION}-query`"


def _reservation_ddl(sql, what):
    try:
        bq.query(sql).result()
        print(f"  created {what}")
        return True
    except Exception as exc:  # noqa: BLE001
        msg = str(exc)
        if "already" in msg.lower():
            print(f"  {what}: already in place")
            return False
        if "bigqueryreservation" in msg or "has not been used" in msg or "is disabled" in msg:
            raise RuntimeError(
                "BigQuery Reservation API is not enabled yet. In Cloud Shell run:\n"
                f"  gcloud services enable bigqueryreservation.googleapis.com --project={PROJECT}\n"
                "then re-run this cell."
            ) from exc
        raise


with step("1b graph-capable Enterprise reservation"):
    if os.environ.get("A4I_SKIP_RESERVATION") == "1":
        print("  skipped reservation DDL (A4I_SKIP_RESERVATION=1)")
    else:
        created = _reservation_ddl(
            f"CREATE RESERVATION {_res_path} "
            f"OPTIONS (edition = 'ENTERPRISE', slot_capacity = 0, autoscale_max_slots = 50)",
            f"reservation {RESERVATION} (ENTERPRISE, 0 baseline slots, autoscale max 50)",
        )
        assigned = _reservation_ddl(
            f"CREATE ASSIGNMENT {_asg_path} "
            f"OPTIONS (assignee = 'projects/{PROJECT}', job_type = 'QUERY')",
            f"QUERY assignment for {PROJECT}",
        )
        if created or assigned:
            print("  waiting 60s for new reservation assignment to propagate...")
            time.sleep(60)
        print(f"  graph queries in {PROJECT} assigned to {RESERVATION}")

## 2. Start with the wrong answer

Suppose a state Part C director asks: **"Where is our referral network breaking the federal 45-day IFSP timeline, and where should we intervene first?"**

The obvious flat-SQL instinct is to sort `pediatric_and_ei_provider_directory` by `active_waitlist_count DESC` and send emergency funding to whichever hub sits at the top. Let's run that flat sort right alongside a multi-hop episode trace on `{STATE}` and watch what flat SQL misses:

- Sorting by `active_waitlist_count` points at saturated downstream `Multidisciplinary_Eval_Hub` queues (waitlist `60–81`), which are **symptoms** at hop 2 and hop 3.
- Tracing complete `case_chain_id` trajectories reveals that `Part_C_Intake_Coordinator` nodes look completely healthy in a flat table (`active_waitlist_count = 20–29`, `median_internal_wait_days = 5–6`), yet **route families via `paper_fax` into already-saturated or non-bilingual evaluation hubs**—triggering 3- and 4-hop re-referral bounces (`70–75` total wait days, blowing past the **45-day federal statutory ceiling**)!

In [ ]:
with step("section 2 flat-waitlist vs multi-hop episode hook"):
    _prov = read_starter_parquet("pediatric_and_ei_provider_directory.parquet")
    _logs = read_starter_parquet("deidentified_part_c_referral_logs.parquet")
    _sp = _prov[_prov["state_code"] == STATE].copy()
    _sl = _logs[_logs["state_code"] == STATE].copy()

    print(f"1) Naive flat sort — top 5 providers in {STATE} by active_waitlist_count:")
    display(
        _sp.sort_values("active_waitlist_count", ascending=False)[
            [
                "provider_id",
                "node_role",
                "active_waitlist_count",
                "median_internal_wait_days",
                "bilingual_services_listed",
            ]
        ].head(5)
    )

    _episodes = (
        _sl.groupby("case_chain_id")
        .agg(total_hops=("hop_sequence", "count"), episode_wait_days=("handoff_wait_days", "sum"))
        .reset_index()
    )
    _hop1 = _sl[_sl["hop_sequence"] == 1].merge(_episodes, on="case_chain_id")
    _coord_impact = (
        _hop1.groupby("target_provider_id")
        .agg(
            routed_episodes=("case_chain_id", "count"),
            chains_3plus_hops=("total_hops", lambda s: int((s >= 3).sum())),
            chains_over_45d=("episode_wait_days", lambda s: int((s > 45).sum())),
            mean_episode_wait_d=("episode_wait_days", lambda s: round(float(s.mean()), 1)),
        )
        .reset_index()
        .merge(
            _sp[["provider_id", "node_role", "active_waitlist_count", "median_internal_wait_days"]],
            left_on="target_provider_id",
            right_on="provider_id",
        )
        .sort_values(["chains_over_45d", "mean_episode_wait_d"], ascending=False)
    )
    print(f"\n2) Multi-hop reality — Part_C_Intake_Coordinator hubs ranked by >45-day breaches:")
    display(
        _coord_impact[
            [
                "provider_id",
                "active_waitlist_count",
                "median_internal_wait_days",
                "routed_episodes",
                "chains_3plus_hops",
                "chains_over_45d",
                "mean_episode_wait_d",
            ]
        ]
    )

Look at the bottom Coordinator row (`SYN-PROV-*-012` / `011`): an internal wait of **5–6 days** and a modest waitlist, yet **9 to 10 of its 11 routed episodes** bounce across 3–4 hops and breach the 45-day IDEA Part C deadline! That multi-hop path dependency is why **BigQuery Property Graphs (`CREATE PROPERTY GRAPH` + ISO `GQL`)** is your differentiator.

## 3. Choosing a state

All six states below carry `30` synthetic `CareNode` organizations and `149` directed `ReferralHandoff` edges across `44` child referral chains (`35` multi-hop chains of `3–4` hops; `26` breaching `45` days), anchored to real Census tracts and regional lead agencies:

- **Leading-zero FIPS (`AL` = `'01'`, `CA` = `'06'`):** Keep `census_tract_geoid` (11 chars) and `county_fips` (5 chars) as strings everywhere.
- **Regional compliance spread (`MS` vs. `CA` / `NY`):** Mississippi's rural Delta agency (`SYN-PARTC-MS-01`) has the lowest timely 45-day IFSP rate in the dataset (`52.5%`), whereas New York's urban Bronx comparator (`SYN-PARTC-NY-03`, `56.2%`) is nearly as strained as its rural northern catchments (`55.2%`).

| State | `FIPS` | Tracts | Children under 5 | Regional 45-day IFSP range | `CareNode`s | Referral chains (`3+` hops / `>45d`) |
|---|---|---:|---:|---|---:|---|
| **`MS`** *(default)* | `28` | 64 | 15,512 | `52.5%` – `65.8%` | 30 | 44 (`35` / `26`) |
| `AL` | `01` | 43 | 9,396 | `54.5%` – `71.6%` | 30 | 44 (`35` / `26`) |
| `GA` | `13` | 57 | 14,505 | `56.4%` – `70.7%` | 30 | 44 (`35` / `26`) |
| `TX` | `48` | 72 | 29,599 | `57.6%` – `70.9%` | 30 | 44 (`35` / `26`) |
| `CA` | `06` | 101 | 41,144 | `61.7%` – `74.9%` | 30 | 44 (`35` / `26`) |
| `NY` | `36` | 89 | 23,511 | `55.2%` – `58.3%` | 30 | 44 (`35` / `26`) |

## 4. Census under-5 tract demographics (`acs_under5_tract_demographics`) — and an all-zero column trap!

This table comes from the **US Census Bureau ACS 5-Year** estimates (`children_under_5_count`, `poverty_rate`, `no_vehicle_rate`, `assistance_rate`, `tract_geom`, and post-output `audit_*` shares).

**Inspect your distributions before writing formulas.** Run `spread()` below and look at `limited_english_rate` (`limited_english_speaking_pop_count`): across all tracts in this ACS extract, **every value is `0.0`**! If your team writes a composite neighborhood triage score like `0.4 * poverty_rate + 0.4 * limited_english_rate + 0.2 * no_vehicle_rate` without checking `spread()`, `40%` of your formula silently collapses to zero—while real language screening disparities sit in `nsch_developmental_screening_survey` (`primary_home_language`) and `pediatric_and_ei_provider_directory` (`bilingual_services_listed`)! Section 12 flags this via `note()` so nobody misses it.

In [ ]:
with step("load acs_under5_tract_demographics"):
    raw_demo = read_starter_parquet("acs_under5_tract_demographics.parquet")
    demo = raw_demo[raw_demo["state_code"] == STATE].copy()
    demo["census_tract_geoid"] = safe_str(demo["census_tract_geoid"], width=11)
    demo["county_fips"] = safe_str(demo["county_fips"], width=5)

    spread(demo["children_under_5_count"], "children_under_5_count")
    spread(demo["poverty_rate"], "poverty_rate")
    spread(demo["no_vehicle_rate"], "no_vehicle_rate")
    spread(demo["limited_english_rate"], "limited_english_rate (TRAP: all 0!)")

    load_table(
        demo,
        "acs_under5_tract_demographics",
        geom_col="tract_geom",
        description=(
            f"US Census ACS 5-Year under-5 population, poverty, zero-vehicle households, "
            f"tract_geom (GEOGRAPHY), and audit_* shares for {STATE_NAME}."
        ),
    )

## 5. Statewide developmental screening strata (`nsch_developmental_screening_survey`) — watch the grain!

Calibrated to the **HRSA National Survey of Children's Health (NSCH)** (`2023–2024 Combined`), this table contains **six rows per state**, and **`geographic_level` is `'Statewide'` on every row**—four Federal Poverty Level (`household_income_fpl_band`) strata and two `primary_home_language` strata.

**Two traps to avoid:**
1. **6× fan-out:** Joining `nsch_developmental_screening_survey` to `acs_under5_tract_demographics` on `state_code` alone multiplies every census tract by **6**!
2. **Ecological fallacy:** NSCH does *not* measure individual census tracts. Cite NSCH for statewide income/language screening context (notice how `Non-English Primary Household Language` screening rates sit roughly half as high as English-primary households!), not as a tract-measured rate.

In [ ]:
with step("load nsch_developmental_screening_survey"):
    raw_nsch = read_starter_parquet("nsch_developmental_screening_survey.parquet")
    nsch = raw_nsch[raw_nsch["state_code"] == STATE].copy()

    display(
        nsch[
            [
                "survey_stratum_id",
                "geographic_level",
                "household_income_fpl_band",
                "primary_home_language",
                "standardized_screening_rate_pct",
                "followup_requested_pct",
                "margin_of_error_pct",
            ]
        ]
    )

    load_table(
        nsch,
        "nsch_developmental_screening_survey",
        description=(
            f"[Modeled Statewide Strata] HRSA NSCH developmental screening (9-35m) and "
            f"follow-up rates by FPL band and household language for {STATE_NAME}."
        ),
    )

## 6. Regional IDEA Part C lead agency performance (`idea_part_c_agency_performance`)

Three **`100% [Synthetic]`** regional Part C lead agencies per state (`SYN-PARTC-*`), calibrated to **US Department of Education Office of Special Education Programs (OSEP)** Part C Indicator C7 timelines (`initial_ifsp_within_45_days_count`, `ifsp_delayed_capacity_count`, `ifsp_delayed_family_count`). Notice that **system capacity delays** (`ifsp_delayed_capacity_count`) outnumber family-reason delays by ~3:1 across rural catchments.

In [ ]:
with step("load idea_part_c_agency_performance"):
    raw_agency = read_starter_parquet("idea_part_c_agency_performance.parquet")
    agencies = raw_agency[raw_agency["state_code"] == STATE].copy()

    display(
        agencies.assign(
            timely_45d_pct=(
                agencies["initial_ifsp_within_45_days_count"]
                / agencies["evaluations_completed_count"].clip(lower=1)
                * 100
            ).round(1)
        )[
            [
                "agency_id",
                "served_counties",
                "annual_referrals_received",
                "evaluations_completed_count",
                "timely_45d_pct",
                "ifsp_delayed_capacity_count",
                "ifsp_delayed_family_count",
            ]
        ]
    )

    load_table(
        agencies,
        "idea_part_c_agency_performance",
        geom_col="intake_geom",
        description=(
            f"[Synthetic] Regional IDEA Part C lead agency 45-day IFSP compliance and delay "
            f"counts with intake_geom (GEOGRAPHY) for {STATE_NAME}."
        ),
    )

## 7. Homogeneous `CareNode` vertex table (`pediatric_and_ei_provider_directory`)

Thirty **`100% [Synthetic]`** organizations per state (`SYN-PROV-*`), structured as a single homogeneous **`CareNode`** vertex table so variable-length ISO GQL paths (`(a:CareNode)-[e:ReferralHandoff]->{2,4}(b:CareNode)`) work cleanly across all four roles:

- `8` `Screening_Pediatric_Clinic` (origin nodes at hop 1)
- `4` `Part_C_Intake_Coordinator` (service coordination hubs)
- `8` `Multidisciplinary_Eval_Hub` (speech/motor/autism evaluation teams—where re-referral bounces happen when bilingual or Medicaid slots are full)
- `10` `EI_Therapy_Vendor` (destination therapy providers delivering IFSP services)

In [ ]:
with step("load pediatric_and_ei_provider_directory"):
    raw_prov = read_starter_parquet("pediatric_and_ei_provider_directory.parquet")
    providers = raw_prov[raw_prov["state_code"] == STATE].copy()
    providers["census_tract_geoid"] = safe_str(providers["census_tract_geoid"], width=11)
    providers["county_fips"] = safe_str(providers["county_fips"], width=5)

    display(
        providers.groupby("node_role")
        .agg(
            nodes=("provider_id", "count"),
            medicaid_nodes=("accepts_medicaid", "sum"),
            bilingual_nodes=("bilingual_services_listed", "sum"),
            median_waitlist=("active_waitlist_count", "median"),
            median_internal_wait_d=("median_internal_wait_days", "median"),
        )
        .reset_index()
    )

    load_table(
        providers,
        "pediatric_and_ei_provider_directory",
        geom_col="provider_geom",
        description=(
            f"[Synthetic] Homogeneous CareNode provider directory (30 fictional organizations "
            f"anchored to real census tracts) with provider_geom (GEOGRAPHY) for {STATE_NAME}."
        ),
    )

## 8. Directed `ReferralHandoff` edge table (`deidentified_part_c_referral_logs`)

`149` directed handoff events across `44` de-identified child referral episodes (`case_chain_id`) in `{STATE}`:

- **2-hop fast-track episodes:** `Screening_Pediatric_Clinic -> Part_C_Intake_Coordinator -> EI_Therapy_Vendor`
- **3-hop standard evaluation episodes:** `Clinic -> Coordinator -> Multidisciplinary_Eval_Hub -> EI_Therapy_Vendor`
- **4-hop re-referral bounce episodes:** `Clinic -> Coordinator -> Eval_Hub_A -> Eval_Hub_B -> EI_Therapy_Vendor` (`Re-Referral Bounce: Evaluator Waitlist Saturation / Bilingual Slot Unavailable`)

Notice below how `paper_fax` handoffs average **~19.5 wait days** and **~24.3% family drop-off**, compared with **~13.3 days** and **~10.1% drop-off** on `ehr_electronic_portal`!

In [ ]:
with step("load deidentified_part_c_referral_logs"):
    raw_logs = read_starter_parquet("deidentified_part_c_referral_logs.parquet")
    logs = raw_logs[raw_logs["state_code"] == STATE].copy()
    logs["child_home_tract_geoid"] = safe_str(logs["child_home_tract_geoid"], width=11)
    logs["referral_date"] = pd.to_datetime(logs["referral_date"]).dt.date

    display(
        logs.groupby("transmission_channel")
        .agg(
            handoffs=("referral_event_id", "count"),
            mean_wait_days=("handoff_wait_days", lambda s: round(float(s.mean()), 1)),
            mean_dropoff_pct=("dropoff_rate_pct", lambda s: round(float(s.mean() * 100), 1)),
        )
        .reset_index()
    )

    load_table(
        logs,
        "deidentified_part_c_referral_logs",
        description=(
            f"[Synthetic] Directed multi-hop ReferralHandoff edge logs (2-4 hops per "
            f"case_chain_id) for {STATE_NAME}."
        ),
    )

## 9. Inventory what landed in BigQuery

In [ ]:
with step("inventory loaded BigQuery tables"):
    inv = bq.query(f"""
        SELECT
            table_id,
            row_count,
            ROUND(size_bytes / 1024.0, 1) AS size_kb
        FROM `{PROJECT}.{DATASET}.__TABLES__`
        ORDER BY table_id
    """).to_dataframe()
    display(inv)

## 10. Episode breach & spatial bilingual-access payoff queries

Before writing GQL in Section 13, let's quantify two foundational truths in SQL:

1. **How hop depth drives federal 45-day IDEA Part C breaches:** 2-hop chains never breach 45 days; 3-hop chains sit right on the razor's edge; 4-hop re-referral bounce chains breach **100% of the time**.
2. **Where high-poverty under-5 neighborhoods sit farthest from a bilingual, Medicaid-accepting `Multidisciplinary_Eval_Hub`:** combining `ST_CENTROID(tract_geom)` with `provider_geom`.

In [ ]:
with step("episode hop-depth & bilingual distance payoff queries"):
    hop_breaches = bq.query(f"""
        WITH episodes AS (
            SELECT
                case_chain_id,
                COUNT(*) AS hops,
                SUM(handoff_wait_days) AS total_wait_days,
                COUNTIF(transmission_channel = 'paper_fax') AS fax_hops,
                MAX(IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1, 0)) AS had_bounce
            FROM `{PROJECT}.{DATASET}.deidentified_part_c_referral_logs`
            GROUP BY case_chain_id
        )
        SELECT
            hops,
            COUNT(*) AS episodes,
            ROUND(AVG(total_wait_days), 1) AS avg_episode_wait_days,
            MIN(total_wait_days) AS min_wait_d,
            MAX(total_wait_days) AS max_wait_d,
            COUNTIF(total_wait_days > 45) AS breaches_over_45d,
            SUM(had_bounce) AS bounce_episodes,
            ROUND(AVG(fax_hops), 1) AS avg_fax_hops
        FROM episodes
        GROUP BY hops
        ORDER BY hops
    """).to_dataframe()
    print("IDEA Part C 45-day compliance by referral chain length:")
    display(hop_breaches)

    eval_gaps = bq.query(f"""
        SELECT
            d.census_tract_geoid,
            d.county_name,
            d.children_under_5_count,
            ROUND(d.poverty_rate * 100, 1) AS poverty_pct,
            ROUND(d.no_vehicle_rate * 100, 1) AS no_vehicle_pct,
            ROUND(
                MIN(ST_DISTANCE(ST_CENTROID(d.tract_geom), p.provider_geom)) / 1000.0,
                1
            ) AS km_to_bilingual_medicaid_eval_hub
        FROM `{PROJECT}.{DATASET}.acs_under5_tract_demographics` AS d
        CROSS JOIN `{PROJECT}.{DATASET}.pediatric_and_ei_provider_directory` AS p
        WHERE p.node_role = 'Multidisciplinary_Eval_Hub'
          AND p.accepts_medicaid = TRUE
          AND p.bilingual_services_listed = TRUE
        GROUP BY 1, 2, 3, 4, 5
        ORDER BY km_to_bilingual_medicaid_eval_hub DESC, d.children_under_5_count DESC
        LIMIT 8
    """).to_dataframe()
    print("\nUnder-5 tracts farthest from a bilingual, Medicaid-accepting Evaluation Hub:")
    display(eval_gaps)

## 11. Deliberate exclusions & the Post-Output Equity Audit rule

1. **`audit_*` columns (`audit_black_share`, `audit_hispanic_share`, `audit_asian_share`, `audit_indigenous_share`, `audit_white_share`) are strictly banned from routing rules, priority scores, and LLM triage prompts.** Route children and prioritize network investments strictly on clinical referral urgency (`child_age_months` approaching the 36-month Part C age-out cliff!), wait-time bottlenecks, bilingual/Medicaid slot capacity, poverty, and travel distance.
2. **Join `audit_*` *after* generating your routing & capacity recommendations** to verify whether children living in high-poverty, zero-vehicle, and historically underserved communities experience disproportionate `>45-day` referral chains—and whether your proposed fix closes that gap.

In [ ]:
with step("post-output equity audit on >45d breached home tracts"):
    breach_equity = bq.query(f"""
        WITH chain_totals AS (
            SELECT
                case_chain_id,
                ANY_VALUE(child_home_tract_geoid) AS child_home_tract_geoid,
                SUM(handoff_wait_days) AS episode_wait_days
            FROM `{PROJECT}.{DATASET}.deidentified_part_c_referral_logs`
            GROUP BY case_chain_id
        )
        SELECT
            IF(c.episode_wait_days > 45, 'Breached (>45 days)', 'Timely (<=45 days)') AS ifsp_status,
            COUNT(*) AS episodes,
            ROUND(AVG(d.poverty_rate) * 100, 1) AS avg_tract_poverty_pct,
            ROUND(AVG(d.no_vehicle_rate) * 100, 1) AS avg_no_vehicle_pct,
            ROUND(AVG(d.audit_black_share) * 100, 1) AS audit_black_pct,
            ROUND(AVG(d.audit_hispanic_share) * 100, 1) AS audit_hispanic_pct,
            ROUND(AVG(d.audit_white_share) * 100, 1) AS audit_white_pct
        FROM chain_totals AS c
        INNER JOIN `{PROJECT}.{DATASET}.acs_under5_tract_demographics` AS d
            ON c.child_home_tract_geoid = d.census_tract_geoid
        GROUP BY 1
        ORDER BY 1
    """).to_dataframe()
    display(breach_equity)

## 12. Validation — graded against the loaded BigQuery tables

Pay special attention to the **graph key & endpoint integrity checks** below: duplicate vertex/edge keys or dangling endpoint references silently corrupt or break `CREATE PROPERTY GRAPH` in Section 13.

In [ ]:
with step("validate loaded BigQuery tables"):
    S = f"{PROJECT}.{DATASET}"

    d_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n,
            COUNT(DISTINCT census_tract_geoid) AS n_distinct,
            COUNTIF(LENGTH(census_tract_geoid) != 11) AS bad_geoid_len,
            COUNTIF(SUBSTR(census_tract_geoid, 1, 2) != '{STATE_FIPS}') AS bad_fips_prefix,
            COUNTIF(state_code != '{STATE}') AS wrong_state,
            COUNTIF(children_under_5_count <= 0) AS nonpos_u5,
            COUNTIF(limited_english_rate = 0) AS zero_lep,
            COUNTIF(
                audit_black_share < 0 OR audit_black_share > 1
                OR audit_hispanic_share < 0 OR audit_hispanic_share > 1
                OR audit_asian_share < 0 OR audit_asian_share > 1
                OR audit_indigenous_share < 0 OR audit_indigenous_share > 1
                OR audit_white_share < 0 OR audit_white_share > 1
            ) AS bad_audit
        FROM `{S}.acs_under5_tract_demographics`
    """).result())[0]

    check("demographics: >= 40 tracts loaded", d_stats.n >= 40, f"{d_stats.n} tracts")
    check("demographics: unique census_tract_geoid", d_stats.n == d_stats.n_distinct, f"{d_stats.n_distinct} distinct")
    check("demographics: all tract IDs are 11 chars (leading zero intact)", d_stats.bad_geoid_len == 0, f"{d_stats.bad_geoid_len} bad")
    check("demographics: all tracts match STATE_FIPS", d_stats.bad_fips_prefix == 0, f"expected {STATE_FIPS}")
    check("demographics: all rows match STATE", d_stats.wrong_state == 0, f"{d_stats.wrong_state} wrong")
    check("demographics: positive children_under_5_count", d_stats.nonpos_u5 == 0, f"{d_stats.nonpos_u5} <= 0")
    check("demographics: audit_* shares within [0, 1]", d_stats.bad_audit == 0, f"{d_stats.bad_audit} out of bounds")

    note(
        "ACS extract note: limited_english_rate is 0.0 across tracts (use NSCH language strata & provider bilingual_services_listed)",
        d_stats.zero_lep,
        d_stats.n,
        detail=f"{d_stats.zero_lep}/{d_stats.n} rows carry 0.0 in limited_english_rate — do not weight silently as non-zero",
        limit=1.01,
    )

    na_stats = list(bq.query(f"""
        SELECT
            (SELECT COUNT(*) FROM `{S}.nsch_developmental_screening_survey`) AS n_nsch,
            (SELECT COUNTIF(geographic_level != 'Statewide' OR state_code != '{STATE}')
               FROM `{S}.nsch_developmental_screening_survey`) AS bad_nsch,
            (SELECT COUNT(*) FROM `{S}.idea_part_c_agency_performance`) AS n_agency,
            (SELECT COUNTIF(NOT STARTS_WITH(agency_id, 'SYN-PARTC-')
                            OR NOT STRPOS(synthetic_agency_name, '[Synthetic]') > 0
                            OR state_code != '{STATE}')
               FROM `{S}.idea_part_c_agency_performance`) AS bad_agency
    """).result())[0]

    check("nsch: exact 6 statewide strata loaded", na_stats.n_nsch == 6 and na_stats.bad_nsch == 0, f"{na_stats.n_nsch} statewide strata")
    check("agencies: 3 [Synthetic] SYN-PARTC-* regional agencies loaded", na_stats.n_agency == 3 and na_stats.bad_agency == 0, f"{na_stats.n_agency} synthetic agencies")

    p_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n_prov,
            COUNT(DISTINCT provider_id) AS n_distinct,
            COUNT(DISTINCT node_role) AS n_roles,
            COUNTIF(state_code != '{STATE}') AS wrong_state,
            COUNTIF(NOT STARTS_WITH(provider_id, 'SYN-PROV-')) AS non_syn_id,
            COUNTIF(STRPOS(synthetic_organization_name, '[Synthetic]') = 0
                    OR STRPOS(synthetic_address, '[Synthetic]') = 0) AS missing_syn_tag,
            COUNTIF(census_tract_geoid NOT IN (
                SELECT census_tract_geoid FROM `{S}.acs_under5_tract_demographics`
            )) AS orphan_prov_tracts
        FROM `{S}.pediatric_and_ei_provider_directory`
    """).result())[0]

    check("providers (CareNode): 30 nodes across all 4 roles", p_stats.n_prov == 30 and p_stats.n_roles == 4, f"{p_stats.n_prov} nodes, {p_stats.n_roles} roles")
    check("providers (CareNode): unique provider_id vertex keys", p_stats.n_prov == p_stats.n_distinct, f"{p_stats.n_distinct} distinct")
    check("providers (CareNode): all rows match STATE", p_stats.wrong_state == 0, f"{p_stats.wrong_state} wrong")
    check("providers (CareNode): 100% stage-safe SYN-PROV-* & [Synthetic] labels", p_stats.non_syn_id == 0 and p_stats.missing_syn_tag == 0, f"non_syn={p_stats.non_syn_id}, untagged={p_stats.missing_syn_tag}")
    check("providers (CareNode): 100% FK match to acs_under5_tract_demographics", p_stats.orphan_prov_tracts == 0, f"{p_stats.orphan_prov_tracts} orphans")

    l_stats = list(bq.query(f"""
        SELECT
            COUNT(*) AS n_edges,
            COUNT(DISTINCT referral_event_id) AS n_distinct_edges,
            COUNT(DISTINCT case_chain_id) AS n_chains,
            COUNTIF(state_code != '{STATE}') AS wrong_state,
            COUNTIF(child_age_months < 0 OR child_age_months > 35) AS bad_age,
            COUNTIF(handoff_wait_days <= 0) AS nonpos_wait,
            COUNTIF(source_provider_id NOT IN (
                SELECT provider_id FROM `{S}.pediatric_and_ei_provider_directory`
            )) AS dangling_src,
            COUNTIF(target_provider_id NOT IN (
                SELECT provider_id FROM `{S}.pediatric_and_ei_provider_directory`
            )) AS dangling_dst,
            COUNTIF(receiving_agency_id NOT IN (
                SELECT agency_id FROM `{S}.idea_part_c_agency_performance`
            )) AS dangling_agency,
            COUNTIF(child_home_tract_geoid NOT IN (
                SELECT census_tract_geoid FROM `{S}.acs_under5_tract_demographics`
            )) AS orphan_home_tract
        FROM `{S}.deidentified_part_c_referral_logs`
    """).result())[0]

    check("referrals (ReferralHandoff): >= 140 edges across >= 40 chains", l_stats.n_edges >= 140 and l_stats.n_chains >= 40, f"{l_stats.n_edges} edges, {l_stats.n_chains} chains")
    check("referrals (ReferralHandoff): unique referral_event_id edge keys", l_stats.n_edges == l_stats.n_distinct_edges, f"{l_stats.n_distinct_edges} distinct")
    check("referrals (ReferralHandoff): all rows match STATE", l_stats.wrong_state == 0, f"{l_stats.wrong_state} wrong")
    check("referrals (ReferralHandoff): Part C age range 0-35 months", l_stats.bad_age == 0, f"{l_stats.bad_age} out of range")
    check("referrals (ReferralHandoff): positive handoff_wait_days", l_stats.nonpos_wait == 0, f"{l_stats.nonpos_wait} <= 0")
    check("referrals (ReferralHandoff): 100% source_provider_id in CareNode", l_stats.dangling_src == 0, f"{l_stats.dangling_src} dangling")
    check("referrals (ReferralHandoff): 100% target_provider_id in CareNode", l_stats.dangling_dst == 0, f"{l_stats.dangling_dst} dangling")
    check("referrals (ReferralHandoff): 100% receiving_agency_id in agencies", l_stats.dangling_agency == 0, f"{l_stats.dangling_agency} dangling")
    check("referrals (ReferralHandoff): 100% child_home_tract_geoid in ACS tracts", l_stats.orphan_home_tract == 0, f"{l_stats.orphan_home_tract} orphans")

    chain_stats = list(bq.query(f"""
        WITH ep AS (
            SELECT
                case_chain_id,
                COUNT(*) AS n_hops,
                MIN(hop_sequence) AS min_hop,
                MAX(hop_sequence) AS max_hop,
                SUM(handoff_wait_days) AS total_wait
            FROM `{S}.deidentified_part_c_referral_logs`
            GROUP BY case_chain_id
        )
        SELECT
            COUNTIF(min_hop != 1 OR max_hop != n_hops) AS broken_sequences,
            COUNTIF(n_hops >= 3) AS multi_hop_3plus,
            COUNTIF(total_wait > 45) AS breached_45d
        FROM ep
    """).result())[0]

    check("chains: contiguous hop_sequence 1..N on every episode", chain_stats.broken_sequences == 0, f"{chain_stats.broken_sequences} broken")
    check("chains: >= 25 multi-hop (3-4 hop) trajectories", chain_stats.multi_hop_3plus >= 25, f"{chain_stats.multi_hop_3plus} chains >= 3 hops")
    check("chains: >= 15 IDEA Part C >45-day breaches to diagnose", chain_stats.breached_45d >= 15, f"{chain_stats.breached_45d} breached chains")

    geo_cols = bq.query(f"""
        SELECT table_name, column_name, data_type
        FROM `{S}.INFORMATION_SCHEMA.COLUMNS`
        WHERE (table_name = 'acs_under5_tract_demographics' AND column_name = 'tract_geom')
           OR (table_name = 'idea_part_c_agency_performance' AND column_name = 'intake_geom')
           OR (table_name = 'pediatric_and_ei_provider_directory' AND column_name = 'provider_geom')
    """).to_dataframe()
    n_geog = int((geo_cols["data_type"] == "GEOGRAPHY").sum()) if len(geo_cols) else 0
    check("spatial: tract_geom, intake_geom, and provider_geom landed as native GEOGRAPHY", n_geog == 3, f"{n_geog}/3 GEOGRAPHY columns")

    failed = [c for c in CHECKS if c["verdict"] == "FAIL"]
    warned = [c for c in CHECKS if c["verdict"] == "WARN"]
    print(f"\nValidation summary: {len(CHECKS)} checks — {len(CHECKS) - len(failed) - len(warned)} PASS, {len(warned)} WARN, {len(failed)} FAIL")
    if failed:
        raise RuntimeError("Validation failed: " + "; ".join(f"{c['check']} ({c['detail']})" for c in failed))

## 13. Framing your differentiator — BigQuery Property Graphs (`CREATE PROPERTY GRAPH` + ISO `GQL`)

This is where the starter notebook stops and your team's graph engineering and agent design take over.

### Step A — Declare the homogeneous `CareNode` Property Graph

Notice **two critical syntax rules** in the DDL below:
1. Table references inside `NODE TABLES` and `EDGE TABLES` are **dataset-qualified and unbackticked** (`a4i_early_intervention.pediatric_and_ei_provider_directory`), while `REFERENCES CareNode (provider_id)` uses the **alias without the dataset**.
2. Exclude `provider_geom` (`PROPERTIES ALL COLUMNS EXCEPT (provider_geom)`) so every graph property is a scalar GQL-compatible type (`latitude` and `longitude` remain on `CareNode`!).

```sql
CREATE OR REPLACE PROPERTY GRAPH a4i_early_intervention.ReferralGraph
  NODE TABLES (
    a4i_early_intervention.pediatric_and_ei_provider_directory AS CareNode
      KEY (provider_id)
      LABEL CareNode
      PROPERTIES ALL COLUMNS EXCEPT (provider_geom)
  )
  EDGE TABLES (
    a4i_early_intervention.deidentified_part_c_referral_logs AS ReferralHandoff
      KEY (referral_event_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL ReferralHandoff
      PROPERTIES ALL COLUMNS
  );
```

### Step B — Variable-length `-{2,4}->` ISO GQL traversal (not Cypher!)

BigQuery implements **ISO/IEC 39075 GQL**—not Neo4j Cypher, and not Gremlin. Look closely at line 6 below (`ALL_DIFFERENT` / `case_chain_id` correlation): without correlating `case_chain_id` across every hop in `hops`, a `-{2,4}->` pattern connects Child A's hop-1 referral into a coordinator with Child B's hop-2 referral out of that coordinator!

```sql
GRAPH a4i_early_intervention.ReferralGraph
MATCH (clinic:CareNode {node_role: 'Screening_Pediatric_Clinic'})
      -[hops:ReferralHandoff]->{2,4}
      (vendor:CareNode {node_role: 'EI_Therapy_Vendor'})
WHERE clinic.state_code = 'MS'
  AND ARRAY_LENGTH(hops) >= 3
  AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
LET total_hops = ARRAY_LENGTH(hops),
    chain_id = hops[OFFSET(0)].case_chain_id,
    chain_wait_days = (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h),
    fax_hops = (SELECT COUNTIF(h.transmission_channel = 'paper_fax') FROM UNNEST(hops) AS h)
FILTER chain_wait_days > 45
RETURN
  chain_id,
  clinic.synthetic_organization_name AS origin_clinic,
  hops[OFFSET(0)].target_provider_id AS intake_coordinator_id,
  vendor.synthetic_organization_name AS final_vendor,
  total_hops,
  fax_hops,
  chain_wait_days
ORDER BY chain_wait_days DESC
LIMIT 15;
```

### Seven BigQuery Graph traps — save yourself 45 minutes

1. **Correlate `case_chain_id` across `UNNEST(hops)` when tracing real child episodes.** Use `(SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1` for historical episode audits—and drop that constraint deliberately *only* when asking "what alternative fast-track paths *could* this coordinator route a new child onto?"
2. **There is no `HAVING` in ISO GQL.** Compute per-path values with `LET` or aggregate in `RETURN`, then chain with `NEXT` + `FILTER`:
   ```sql
   GRAPH a4i_early_intervention.ReferralGraph
   MATCH (c:CareNode {node_role: 'Part_C_Intake_Coordinator'})-[e:ReferralHandoff]->(hub:CareNode)
   RETURN c.provider_id AS coord_id, hub.provider_id AS hub_id, AVG(e.handoff_wait_days) AS avg_wait
   NEXT
   FILTER avg_wait > 18
   RETURN coord_id, hub_id, ROUND(avg_wait, 1) AS avg_wait
   ORDER BY avg_wait DESC;
   ```
3. **Unbackticked `dataset.table` in `CREATE PROPERTY GRAPH`.** Backticking `` `project.dataset.table` `` turns the whole three-part name into one quoted identifier and makes `REFERENCES` fail with *"The referenced node table is not defined"*.
4. **Calling GQL from Python / ADK tools:** Wrap GQL inside standard SQL via `FROM GRAPH_TABLE(a4i_early_intervention.ReferralGraph MATCH ... COLUMNS (...))` or run `GRAPH a4i_early_intervention.ReferralGraph ...` directly via `bq.query(sql)` (works seamlessly once `Section 1b`'s `a4i-graph` reservation is active!).
5. **Visualizing paths requires `ANY SHORTEST` + `TO_JSON(p)`.** Returning bare columns renders a table; returning `TO_JSON(p) AS paths` over `MATCH p = ANY SHORTEST ...` keeps payloads well under Colab's 2 MB graph widget cap.
6. **Watch the 36-month Part C age-out cliff (`child_age_months`).** A 31-month-old toddler stuck in a 75-day fax bounce chain does not merely get a late IFSP—they risk aging out of Part C (`36 months`) before therapy starts!
7. **Spanner Graph docs look almost identical to BigQuery Graph docs.** Verify documentation URLs contain `/bigquery/`, not `/spanner/`.

### Four questions judges will ask from your 3-minute video & deck

1. **Graph necessity:** Show a bottleneck or fast-track reroute your GQL traversal uncovered that flat single-table sorting (`Section 2`) missed.
2. **Actionable routing vs. retrospective blame:** For a newly referred 28-month-old Spanish-speaking Medicaid toddler in a high-poverty tract, which specific `Clinic -> Coordinator -> Eval Hub -> Therapy Vendor` path keeps expected total wait under **45 days**, and why?
3. **Synthetic & grain honesty:** Does your deck/video clearly state that `SYN-PROV-*` / `SYN-PARTC-*` are `100% [Synthetic]` (protecting real clinicians) and that HRSA NSCH (`nsch_developmental_screening_survey`) is `Statewide`-only?
4. **Post-output Equity Audit:** What did joining `audit_*` and `no_vehicle_rate` reveal about which neighborhoods suffer `>45-day` breaches—and how did you address zero-vehicle travel barriers without using race as a routing input?

## 14. Diagnostic summary

In [ ]:
print("=" * 72)
print("A4I EARLY INTERVENTION TRIAGE — DIAGNOSTIC SUMMARY")
print("=" * 72)
print(f"project     : {PROJECT}")
print(f"dataset     : {PROJECT}.{DATASET} ({LOCATION})")
print(f"reservation : {RESERVATION} ({_region})")
print(f"state       : {STATE_NAME} ({STATE}, state FIPS {STATE_FIPS})")
print("-" * 72)
print("TABLES")
for tbl_item in bq.list_tables(f"{PROJECT}.{DATASET}"):
    tbl_obj = bq.get_table(tbl_item)
    cols_preview = ", ".join(f"{f.name}:{f.field_type}" for f in tbl_obj.schema[:6])
    print(f"  {tbl_obj.table_id:<38} {tbl_obj.num_rows:>6,} rows  [{cols_preview}, ...]")
print("-" * 72)
print("CHECKS")
for ck in CHECKS:
    print(f"  {ck['verdict']:<5} {ck['check']}: {ck['detail']}")
print("-" * 72)
print("TIMING")
for s_name, secs in STEPS.items():
    print(f"  {s_name:<44} {secs:>6.1f}s")
print(f"  {'TOTAL WALL CLOCK':<44} {time.time() - NOTEBOOK_START:>6.1f}s")
print("=" * 72)